## SET is_causal = TRUE in Settings

In [1]:
import torch
import torch.nn as nn
from Settings import TinyStoriesLM
from datetime import datetime
import random
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from tokenizer import TinyStoriesTokenizer
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass, asdict
import numpy as np
import os
print("cell ran")

cell ran


In [2]:
# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 1
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


class ProbeDataset(Dataset):
    def __init__(self, data, tensor_tags, index_range, limit=512, ignore_index = -100):
        self.data = data
        self.limit = limit
        self.index_range = list(index_range)
        self.tensor_tags = tensor_tags
        self.ignore_index = ignore_index
        #self.pos_to_id = pos_to_id

    def __len__(self):
        return len(self.index_range)

    def __getitem__(self, idx):
        idx_true = self.index_range[idx]
    
        # when we built tensors, we bactch (0, 512, 1024, 1536...)
        # To get the real position
        dataset_index = idx_true * self.limit

        # Extract binary file
        item = self.data[dataset_index]
        input_ids = item[0][:self.limit] 
        
        # from tensor tags
        labels = self.tensor_tags[idx_true][:self.limit].clone()
        labels[labels < 3] = self.ignore_index
        
        return {
            'input_ids': input_ids,
            'labels': labels
        }

print("probedataset ready")

probedataset ready


In [3]:
# We load the aligned data
tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')
POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
ignore_index = -100
print(f"{len(tensores_tags_dataset)} tensors for Probing.")

5000 tensors for Probing.


In [4]:
import json

with open("multipos_dict.json", "r") as f:
    multipos_dict = json.load(f)

print("cell ran")

cell ran


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = 'last_checkpoint_next_word.pt'

model = TinyStoriesLM.load(checkpoint_path, device=device).to(device)
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')

# We have to freeze all the model
for param in model.parameters():
    param.requires_grad = False

# We set our model i eval mode -> deactivate dropout  so the vectors are stable
# dropout turns off random neurons during training to avoid overfitting/memorizing 
model.eval() #same input = same output

activations = {}

def forward_hook(module, input, output): # hook function
    # 'output' - ouput from Transformer layer
    # module layer that was just executed
    # save in our dictionary
    # .detach() no gradient info / history - we just want numbers
    
    if isinstance(output, tuple):
        activations[module.name] = output[0].detach()
    else:
        activations[module.name] = output.detach()

hooks = []

# register each block (TransformerBlock)
for i, block in enumerate(model.transformers):
    block.name = f"layer_{i}" 
    handle = block.register_forward_hook(forward_hook)
    hooks.append(handle)


num_tags = len(tag2id)
input_dim = model.config.vector_dim
limit = model.config.block_size
num_layers = len(model.transformers)

# from 5,000 stories -> 4500 train and 500 validate
# data for training
train_subset_tags = tensores_tags_dataset[:4500]
val_subset_tags = tensores_tags_dataset[4500:]

# index_range to ensure aligment
train_ds = ProbeDataset(training_dataset, tensores_tags_dataset, index_range=range(0, 4500), limit=limit)
train_loader = DataLoader(train_ds, batch_size=model.config.batch_size, shuffle=True)

val_ds = ProbeDataset(training_dataset, tensores_tags_dataset, index_range=range(4500, 5000), limit=limit)
val_loader = DataLoader(val_ds, batch_size=model.config.batch_size, shuffle=False)


Model loaded from last_checkpoint_next_word.pt (Epoch 0, iteration 130000)


In [6]:
print(f"Subsets : training: {len(train_ds)} | validation: {len(val_ds)}")

output_dir = os.path.join("plts", "next_word")
os.makedirs(output_dir, exist_ok=True)
target_words = [' so', ' playing', ' back', ' it', ' he']
evolution_tracking = {word: {} for word in target_words}

# Global accumulation 
all_general_acc = []
all_ambiguous_acc = []
all_general_pres = []
all_ambiguous_pres = []
all_general_rec = []
all_ambiguous_rec = []
all_general_f1 = []
all_ambiguous_f1 = []

# loop through each layer representation
for layer_to_probe in range(num_layers):
    layer_name = f"layer_{layer_to_probe}"
    print(f"\nTraining probe of: {layer_name}")

    # Reset linear probing for current layer
    probe = nn.Linear(input_dim, num_tags).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    # Epochs
    for epoch in range(3):
        probe.train()
        for batch in train_loader:
            activations.clear()
            ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            with torch.no_grad():
                model(ids) 

            optimizer.zero_grad()
            logits = probe(activations[layer_name])
   
            loss = criterion(logits.view(-1, num_tags), labels.view(-1))
            loss.backward()
            optimizer.step()

        # Validation phase
        probe.eval()
        
        all_preds_general = []
        all_labels_general = []
        all_preds_ambiguous = []
        all_labels_ambiguous = []
        
        with torch.no_grad():
            for v_batch in val_loader:
                v_ids = v_batch['input_ids'].to(device)
                v_labels = v_batch['labels'].to(device)
                
                model(v_ids)
                v_logits = probe(activations[layer_name])
                predictions = v_logits.argmax(dim=-1)

                v_probs = torch.softmax(v_logits, dim=-1)
                top_3_probs, top_3_idx = torch.topk(v_probs, k=3, dim=-1)  

                # Only not ignored elements
                mask = (v_labels != ignore_index)
                all_preds_general.extend(predictions[mask].cpu().tolist())
                all_labels_general.extend(v_labels[mask].cpu().tolist())

                for b in range(v_ids.size(0)):
                    ids_list = v_ids[b].tolist()
                    tokens = tokenizer.decode_to_tokens(ids_list)
                    
                    for t in range(len(tokens)):
                        label_real = v_labels[b, t].item()
                        if label_real == ignore_index:
                            continue 
                        
                        current_word = tokens[t] 
    
                        # Qualitative log (final epoch only)
                        if epoch == 2 and current_word in target_words:
                            inicio_contexto = max(0, t - 5)
                            fin_contexto = min(len(ids_list), t + 5)
                            texto_antes = tokenizer.decode(ids_list[inicio_contexto: t])
                            frase_id = f"{texto_antes} *[{current_word.upper().strip()}]*"

                            if frase_id not in evolution_tracking[current_word]:
                                evolution_tracking[current_word][frase_id] = {}
                            
                            pred_label = predictions[b, t].item()
                            top_3_lista = top_3_idx[b, t].tolist()
                            top_3_tags = [id2tag[idx] for idx in top_3_lista]
                            top_3_probs_list = top_3_probs[b, t].tolist()
                            percentages = [p * 100 for p in top_3_probs_list]
                                
                            evolution_tracking[current_word][frase_id][layer_name] = {
                                "real_tag": id2tag[label_real],
                                "pred_tag": id2tag[pred_label],
                                "top_3_tags": top_3_tags,
                                "percentages": percentages
                            }
                                
                        # Collect ambiguous info
                        if current_word in multipos_dict:
                            pred_label = predictions[b, t].item()
                            all_preds_ambiguous.append(pred_label)
                            all_labels_ambiguous.append(label_real)

        # Filter to strip subwordsl tags (0, 1, 2)
        valid_indices = [i for i, lbl in enumerate(all_labels_general) if lbl >= 3]
        all_labels_general = [all_labels_general[i] for i in valid_indices]
        all_preds_general = [all_preds_general[i] for i in valid_indices]
        unique_valid_labels = sorted(list(set(all_labels_general + all_preds_general)))

        # General evaluations
        epoch_accuracy = accuracy_score(all_labels_general, all_preds_general) * 100
        gen_prec, gen_rec, gen_f1, _ = precision_recall_fscore_support(
            all_labels_general, all_preds_general, labels=unique_valid_labels, average='macro', zero_division=0
        )
        epoch_precision = gen_prec * 100
        epoch_recall = gen_rec * 100
        epoch_f1 = gen_f1 * 100
        
        # Ambiguous subset evaluations (also ignore 0,1,2)
        amb_valid_labels = sorted(list(set([l for l in all_labels_ambiguous + all_preds_ambiguous if l >= 3])))
        if len(all_labels_ambiguous) > 0 and len(amb_valid_labels) > 0:
            epoch_ambiguity_accuracy = accuracy_score(all_labels_ambiguous, all_preds_ambiguous) * 100
            amb_prec, amb_rec, amb_f1, _ = precision_recall_fscore_support(
                all_labels_ambiguous, all_preds_ambiguous, labels = amb_valid_labels, average='macro', zero_division=0
            )
            epoch_amb_precision = amb_prec * 100
            epoch_amb_recall = amb_rec * 100
            epoch_amb_f1 = amb_f1 * 100
        else:
            epoch_ambiguity_accuracy, epoch_amb_precision, epoch_amb_recall, epoch_amb_f1 = 0.0, 0.0, 0.0, 0.0
       
        print(f" Epoch {epoch + 1} [General]   accuracy: {epoch_accuracy:.2f}% | precision: {epoch_precision:.2f}% | recall: {epoch_recall:.2f}% | F1: {epoch_f1:.2f}%")
        print(f" Epoch {epoch + 1} [Ambiguous] accuracy: {epoch_ambiguity_accuracy:.2f}% | precision: {epoch_amb_precision:.2f}% | recall: {epoch_amb_recall:.2f}% | F1: {epoch_amb_f1:.2f}%")
        
        # confusion data to numpy arrays
        if epoch == 2:
            tag_names = [id2tag[idx] for idx in unique_valid_labels]
            cm = confusion_matrix(all_labels_general, all_preds_general, labels=unique_valid_labels, normalize='true')
            matrix_data = {
                "matrix": cm,
                "tag_names": tag_names,
                "tag_ids": unique_valid_labels
            }
            
            matrix_path = os.path.join(output_dir, f'matrix_data_{layer_name}.npy')
            np.save(matrix_path, matrix_data, allow_pickle=True)
            print(f" -- Saved matrix array data to: {matrix_path} --")

    # Log for line plot 
    all_general_acc.append(epoch_accuracy)
    all_ambiguous_acc.append(epoch_ambiguity_accuracy)
    all_general_pres.append(epoch_precision)
    all_ambiguous_pres.append(epoch_amb_precision)
    all_general_rec.append(epoch_recall)
    all_ambiguous_rec.append(epoch_amb_recall)
    all_general_f1.append(epoch_f1)
    all_ambiguous_f1.append(epoch_amb_f1)
    
    print(f"="*70) 

# Disconnect hooks
for handle in hooks:
    handle.remove()

# metrics to json
experiment_results = {
    "metadata": {
        "num_layers": num_layers,
        "target_words": target_words
    },
    "general_metrics": {
        "layers": [f"layer_{i}" for i in range(num_layers)],
        "accuracy": all_general_acc,
        "precision": all_general_pres,
        "recall": all_general_rec,
        "f1_score": all_general_f1
    },
    "ambiguous_metrics": {
        "accuracy": all_ambiguous_acc,
        "precision": all_ambiguous_pres,
        "recall": all_ambiguous_rec,
        "f1_score": all_ambiguous_f1
    },
    "evolution_tracking": evolution_tracking
}

json_path = os.path.join(output_dir, "probing_results.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(experiment_results, f, ensure_ascii=False, indent=4)

print("cell ran")

Subsets : training: 4500 | validation: 500

Training probe of: layer_0
 Epoch 1 [General]   accuracy: 96.17% | precision: 94.86% | recall: 95.12% | F1: 94.96%
 Epoch 1 [Ambiguous] accuracy: 94.72% | precision: 91.76% | recall: 93.04% | F1: 92.28%
 Epoch 2 [General]   accuracy: 96.55% | precision: 95.40% | recall: 95.68% | F1: 95.53%
 Epoch 2 [Ambiguous] accuracy: 95.21% | precision: 92.54% | recall: 93.51% | F1: 92.97%
 Epoch 3 [General]   accuracy: 96.76% | precision: 95.81% | recall: 95.86% | F1: 95.83%
 Epoch 3 [Ambiguous] accuracy: 95.48% | precision: 93.17% | recall: 93.65% | F1: 93.39%
 -- Saved matrix array data to: plts/next_word/matrix_data_layer_0.npy --

Training probe of: layer_1
 Epoch 1 [General]   accuracy: 97.08% | precision: 96.33% | recall: 95.99% | F1: 96.15%
 Epoch 1 [Ambiguous] accuracy: 96.06% | precision: 94.28% | recall: 94.01% | F1: 94.12%
 Epoch 2 [General]   accuracy: 97.34% | precision: 96.71% | recall: 96.47% | F1: 96.59%
 Epoch 2 [Ambiguous] accuracy: 96.3

In [7]:
fixed_rand_phrases = {}

for word, contexts in evolution_tracking.items():
    phrase_list = list(contexts.keys())
    phrase_list = sorted(phrase_list) # base order
    random.seed(103)
    num_elements = min(5, len(phrase_list)) # if any element has less than 5
    selected_phrases = random.sample(phrase_list, num_elements)
    fixed_rand_phrases[word] = selected_phrases

for word, phrases in fixed_rand_phrases.items():
    print("-"*50)
    print(f"Ambiguous Word: {repr(word)}")
    for phrase_id in phrases: # each dif phrase from that ambiguous word
       layers = evolution_tracking[word][phrase_id]
       print("-"*50)
       print(f"Sentence ... {phrase_id}") 
       for layer_name in sorted(layers.keys()):
           info = layers[layer_name]
           print(f"{layer_name} | Pred: {info['pred_tag']} | True: {info['real_tag']}")
           print(f" Top 1: {info['top_3_tags'][0]} ({info['percentages'][0]:.1f}%) | Top 2: {info['top_3_tags'][1]} ({info['percentages'][1]:.1f}%) | Top 3: {info['top_3_tags'][2]} ({info['percentages'][2]:.1f}%)")
            
    print("")

--------------------------------------------------
Ambiguous Word: ' so'
--------------------------------------------------
Sentence ...  to play. They had *[SO]*
layer_0 | Pred: ADV | True: ADV
 Top 1: ADV (92.8%) | Top 2: SCONJ (4.8%) | Top 3: VERB (0.9%)
layer_1 | Pred: ADV | True: ADV
 Top 1: ADV (94.8%) | Top 2: SCONJ (2.5%) | Top 3: VERB (0.7%)
layer_2 | Pred: ADV | True: ADV
 Top 1: ADV (96.8%) | Top 2: SCONJ (2.0%) | Top 3: PRON (0.5%)
layer_3 | Pred: ADV | True: ADV
 Top 1: ADV (94.7%) | Top 2: NOUN (2.2%) | Top 3: ADP (1.1%)
--------------------------------------------------
Sentence ...  not like the noise, *[SO]*
layer_0 | Pred: ADV | True: CCONJ
 Top 1: ADV (90.3%) | Top 2: SCONJ (6.5%) | Top 3: CCONJ (0.9%)
layer_1 | Pred: ADV | True: CCONJ
 Top 1: ADV (90.8%) | Top 2: CCONJ (4.1%) | Top 3: SCONJ (2.8%)
layer_2 | Pred: ADV | True: CCONJ
 Top 1: ADV (83.2%) | Top 2: SCONJ (9.8%) | Top 3: VERB (1.8%)
layer_3 | Pred: ADV | True: CCONJ
 Top 1: ADV (56.8%) | Top 2: SCONJ (22.4

In [8]:
for word, contexts in evolution_tracking.items():
    print("=" * 80)
    print(f"SEARCHING HIGH-UNCERTAINTY CASES FOR: {repr(word)}")
    print("=" * 80)
    
    match_count = 0
    
    for phrase_id, layers in contexts.items():
        # Verificamos si en AL MENOS una capa el Top 2 superó el 20%
        has_high_uncertainty = any(layers[layer]['percentages'][1] > 20.0 for layer in layers)
        
        if has_high_uncertainty:
            match_count += 1
            print(f"\n[Case #{match_count}] Sentence: {phrase_id}")
            print("-" * 60)
            
            # Mostramos cómo evolucionó la duda capa por capa en esta frase
            for layer_name in sorted(layers.keys()):
                info = layers[layer_name]
                
                # Alerta visual si esta capa específica es la que causó la duda
                alert = "⚠️" if info['percentages'][1] > 20.0 else "   "
                
                print(f"  {layer_name} | Pred: {info['pred_tag']} | True: {info['real_tag']} {alert}")
                print(f"    Top 1: {info['top_3_tags'][0]} ({info['percentages'][0]:.1f}%) | "
                      f"Top 2: {info['top_3_tags'][1]} ({info['percentages'][1]:.1f}%) | ")
            print("-" * 60)
            
    if match_count == 0:
        print(f"No rare cases found for {repr(word)} with Top 2 > 10% in this run.\n")
    else:
        print(f"✅ Found {match_count} interesting cases for {repr(word)}.\n")

SEARCHING HIGH-UNCERTAINTY CASES FOR: ' so'

[Case #1] Sentence:  for it to breathe, *[SO]*
------------------------------------------------------------
  layer_0 | Pred: ADV | True: ADV ⚠️
    Top 1: ADV (75.2%) | Top 2: SCONJ (22.3%) | 
  layer_1 | Pred: ADV | True: ADV ⚠️
    Top 1: ADV (68.1%) | Top 2: SCONJ (25.4%) | 
  layer_2 | Pred: ADV | True: ADV ⚠️
    Top 1: ADV (68.1%) | Top 2: SCONJ (28.3%) | 
  layer_3 | Pred: ADV | True: ADV    
    Top 1: ADV (82.4%) | Top 2: SCONJ (13.6%) | 
------------------------------------------------------------

[Case #2] Sentence:  was an ancient symbol, *[SO]*
------------------------------------------------------------
  layer_0 | Pred: ADV | True: ADV    
    Top 1: ADV (88.4%) | Top 2: SCONJ (9.6%) | 
  layer_1 | Pred: ADV | True: ADV ⚠️
    Top 1: ADV (72.9%) | Top 2: SCONJ (22.7%) | 
  layer_2 | Pred: ADV | True: ADV    
    Top 1: ADV (91.7%) | Top 2: SCONJ (6.8%) | 
  layer_3 | Pred: ADV | True: ADV    
    Top 1: ADV (93.4%) | Top 2: 